# 04 — Effective Receptive Field (ERF) · Thành viên 3

**Câu hỏi:** Khi xét một đặc trưng ở giữa mạng, những pixel nào của ảnh ảnh hưởng
mạnh đến đặc trưng đó? Phân bố này thay đổi thế nào qua các tầng của ResNet-50 và ViT-B/16?

Notebook này tự chứa code, chạy được trên Colab hoặc Jupyter. Bạn không cần train model.
Hãy đọc phần giải thích rồi chạy từng ô, chưa dùng **Run all** trong lần đầu.

Lộ trình: **GPU → dữ liệu → một ảnh → gradient → metrics → nhiều ảnh → báo cáo**.
Code nguồn tương ứng: `src/erf.py`; notebook được tạo bằng `src/make_erf_notebook.py`.
Các ô chưa được chạy sẵn; không có số liệu ERF ImageNet giả lập trong notebook.

## Bước 0 — Kiểm tra nơi chạy

Trên [Google Colab](https://colab.research.google.com/), chọn **File → Upload notebook**,
mở file này, rồi vào **Runtime → Change runtime type → chọn GPU** nếu được cấp.
Chạy ô dưới. `CUDA: True` nghĩa là PyTorch nhìn thấy GPU.
`False` vẫn chạy thử được bằng CPU, nhưng hãy kiểm tra lại runtime trước khi chạy nhiều ảnh.

Colab thường có sẵn torch/torchvision. Nếu thiếu thư viện phụ, ô này cài vào đúng kernel.
Khi dùng máy cá nhân, cài bộ torch/torchvision phù hợp theo
[hướng dẫn PyTorch](https://pytorch.org/get-started/locally/) trước.

In [ ]:
import importlib.util
import subprocess
import sys

missing = [package for package in ['numpy', 'pandas', 'matplotlib', 'PIL', 'pyarrow', 'tqdm']
           if importlib.util.find_spec(package) is None]
if missing:
    packages = ['pillow' if name == 'PIL' else name for name in missing]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *packages])

import torch
import torchvision

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA:', torch.cuda.is_available())
print('Thiết bị:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU')

In [ ]:
from collections import Counter
from io import BytesIO
import hashlib
import json
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import pyarrow.parquet as pq
import torch
import torchvision
from torchvision.models import (
    ResNet50_Weights, ViT_B_16_Weights, resnet50, vit_b_16,
)
from tqdm.auto import tqdm


MODEL_SPECS = {
    "resnet50": (resnet50, ResNet50_Weights.IMAGENET1K_V2),
    "vit_b_16": (vit_b_16, ViT_B_16_Weights.IMAGENET1K_V1),
}
LAYERS = {
    "resnet50": ["layer1", "layer2", "layer3", "layer4"],
    "vit_b_16": ["block3", "block6", "block9", "block12"],
}

## Bước 1 — Kết nối đúng dữ liệu của nhóm

GitHub chứa **danh sách ảnh**, không chứa ảnh ImageNet. Cần các file sau trong cùng
thư mục project (lấy bộ validation mà nhóm đã chuẩn bị, không đổi sang bộ ảnh khác):

```text
cnn-vit-interpretability/
  configs/analysis_set.json
  data/manifests/analysis_10000_seed42.csv
  data/imagenet/data/validation-00000-of-00014.parquet
  ... đến validation-00013-of-00014.parquet
```

**Colab:** đặt thư mục này trong Google Drive, chạy ô mount rồi sửa đường dẫn nếu cần.
Đọc trực tiếp các shard từ Drive thuận tiện cho lần thử đầu. Khi I/O chậm, có thể copy
các shard vào `/content` và đặt lại `DATA_DIR`; giữ `OUTPUT_BASE` trên Drive để lưu kết quả.
Runtime Colab là tạm thời; file chỉ nằm trong `/content` có thể mất khi runtime bị thu hồi.

**Máy cá nhân:** mở notebook từ thư mục repo hoặc `notebooks/`; ô cấu hình tự tìm project.
Kiểm tra thư mục đúng trước khi chạy tiếp. Bước này không tải model.

In [ ]:
IS_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
if IS_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/cnn-vit-interpretability')
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / 'configs/analysis_set.json').is_file()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError('Mở notebook từ repo hoặc sửa PROJECT_ROOT bằng đường dẫn đầy đủ.')

DATA_DIR = PROJECT_ROOT / 'data/imagenet/data'
OUTPUT_BASE = PROJECT_ROOT / 'results/metrics/erf'
print('Project:', PROJECT_ROOT)
print('Số shard validation:', len(list(DATA_DIR.glob('validation-*.parquet'))), '/ 14')
print('Config:', (PROJECT_ROOT / 'configs/analysis_set.json').is_file())
print('Manifest:', (PROJECT_ROOT / 'data/manifests/analysis_10000_seed42.csv').is_file())

## Bước 2 — Chọn một tập ảnh nhỏ, dùng chung cho hai model

Ta kiểm tra SHA-256 của manifest gốc rồi lấy ngẫu nhiên 10 ảnh bằng seed 42.
Không lấy 10 dòng đầu vì manifest sắp theo ID, có thể tạo thiên lệch.
Đây là lấy mẫu đều theo ảnh, **không phải** tập con cân bằng đủ 1.000 lớp.
Danh sách được lưu cùng kết quả, để biết chính xác đã dùng ảnh nào.

Trong lần đầu giữ `NUM_IMAGES = 10`; khi chạy ổn mới đổi thành 100–200.
Để xác minh đầy đủ revision của các shard, chạy script có sẵn
`src/create_analysis_manifest.py --check` tại project; phép kiểm tra notebook
kiểm tra checksum manifest và ID/label/index của các ảnh thực sự được đọc.

In [ ]:
def select_records(project_root, num_images=10, seed=42):
    """Sample uniformly from the approved manifest, never from its ID prefix."""
    root = Path(project_root)
    config = json.loads((root / "configs/analysis_set.json").read_text("utf-8"))
    manifest = root / config["manifest"]
    manifest_bytes = manifest.read_bytes().replace(b"\r\n", b"\n")
    if hashlib.sha256(manifest_bytes).hexdigest() != config["manifest_sha256"]:
        raise ValueError("Analysis manifest checksum differs from the shared config.")
    frame = pd.read_csv(manifest)
    counts = Counter(frame.label)
    if (len(frame) != config["num_images"] or not frame.image_id.is_unique
            or not frame.dataset_index.is_unique
            or set(counts) != set(range(config["num_classes"]))
            or set(counts.values()) != {config["images_per_class"]}):
        raise ValueError("Expected 10,000 unique images, exactly 10 per class.")
    if not 1 <= num_images <= len(frame):
        raise ValueError("num_images must be between 1 and 10,000.")
    records = frame.sort_values("image_id").to_dict("records")
    chosen = random.Random(seed).sample(records, num_images)
    return sorted(chosen, key=lambda row: row["dataset_index"])


def load_images(data_dir, records):
    """Stream Parquet; decode only requested IDs and verify their labels/indices."""
    files = sorted(Path(data_dir).glob("validation-*.parquet"))
    if len(files) != 14:
        raise FileNotFoundError(
            f"Expected 14 ImageNet validation shards in {data_dir}. "
            "The repository contains IDs, not the image bytes."
        )
    wanted = {row["image_id"]: row for row in records}
    if len(wanted) != len(records):
        raise ValueError("Duplicate selected image IDs.")
    found = set()
    offset = 0
    for path in files:
        parquet = pq.ParquetFile(path)
        # Skip entire shards that cannot contain any requested global index.
        end = offset + parquet.metadata.num_rows
        if not any(offset <= row["dataset_index"] < end for row in records):
            offset = end
            continue
        for batch in parquet.iter_batches(batch_size=32, columns=["image", "label"]):
            for row in batch.to_pylist():
                index = offset
                offset += 1
                image_id = row["image"]["path"]
                if image_id not in wanted:
                    continue
                expected = wanted[image_id]
                if (image_id in found or row["label"] != expected["label"]
                        or index != expected["dataset_index"]):
                    raise ValueError(f"Dataset/selection mismatch: {image_id}")
                with Image.open(BytesIO(row["image"]["bytes"])) as image:
                    rgb = image.convert("RGB")
                found.add(image_id)
                yield expected, rgb
        if len(found) == len(wanted):
            break
    if found != set(wanted):
        raise ValueError(f"Missing selected images: {sorted(set(wanted) - found)[:5]}")

In [ ]:
SEED = 42
NUM_IMAGES = 10
PREPROCESSING = 'official'
records = select_records(PROJECT_ROOT, NUM_IMAGES, SEED)
display(pd.DataFrame(records))
print('Số lớp trong tập nhỏ:', len({row['label'] for row in records}))

In [ ]:
images = load_images(DATA_DIR, records)
example_record, example_image = next(images)
images.close()
print('Ảnh thử:', example_record)
plt.figure(figsize=(5, 5))
plt.imshow(example_image)
plt.axis('off')
plt.show()

## Bước 3 — Chọn đặc trưng cần khảo sát

| Giai đoạn tương đối | ResNet-50 V2 | ViT-B/16 V1 |
|---|---|---|
| 1 | layer1 | block 3 |
| 2 | layer2 | block 6 |
| 3 | layer3 | block 9 |
| 4 | layer4 | block 12 |

Đây là các mốc theo độ sâu, **không phải** các tầng tương đương về tính toán.
Với ResNet, lấy vector kênh ở giữa feature map, sau toàn bộ stage.
Với ViT, lấy vector của patch token ở gần tâm, sau toàn bộ block và trước
LayerNorm cuối của encoder. CLS vẫn tham gia attention nhưng không được chọn làm đầu ra.

Quy ước chỉ số bắt đầu từ 0: `row=H//2`, `col=W//2`.
Với lưới patch 14×14: chọn (7,7), token thứ `1 + 7*14 + 7 = 106` trong chuỗi có CLS.
Lưới chẵn không có duy nhất một ô tâm; vị trí receptive center giữa hai kiến trúc
cũng không trùng hoàn toàn. Notebook lưu vị trí cụ thể trong CSV.

Mỗi vector có nhiều số, cần chọn **một scalar** để backward.
Ta dùng $z=\frac{1}{C}\sum_k f_k^2$ (năng lượng trung bình của vector),
cùng quy tắc cho cả hai model. Đây là một lựa chọn thực nghiệm có chủ đích;
kết quả phụ thuộc lựa chọn scalar và không phải chuẩn toàn bộ Jacobian.
Không dùng trung bình kênh sau LayerNorm vì có thể gần hằng số và triệt tiêu gradient.

## Bước 4 — Từ gradient thành bản đồ ERF

Với $u$ là RGB trong [0,1] **sau resize/crop**, ta đo:

$$ E_{ij}=\sum_{c=1}^3\left|\frac{\partial z}{\partial u_{cij}}\right|,
   \qquad P_{ij}=E_{ij}/\sum_{ab}E_{ab}. $$

`requires_grad_(True)` yêu cầu PyTorch theo dõi ảnh đầu vào;
`autograd.grad(z, x)` tính độ nhạy của scalar với từng pixel.
`eval()` tắt dropout và dùng BatchNorm đã học, vẫn cho phép gradient.
Không bọc ERF trong `inference_mode()`.

Model nhận $x_c=(u_c-\mu_c)/\sigma_c$, nên phải chia gradient theo $x_c$
cho $\sigma_c$ để đổi về gradient theo RGB. Ta không tính đạo hàm ngược
qua phép resize/crop về ảnh gốc. Gradient thô được lưu riêng; bản đồ chuẩn hóa
mô tả phân bố tương đối. Tổng gradient thô không dùng để xếp hạng hai kiến trúc.

Đây là phép đo độ nhạy cục bộ của biểu diễn, không chứng minh pixel đó gây ra
quyết định phân loại. Cách scalar hóa và trị tuyệt đối là lựa chọn của notebook,
không phải sao chép nguyên xi mọi thiết lập của Luo et al. (2016).

In [ ]:
def center_features(model, model_name, x):
    """Return center vectors AFTER each complete residual stage/block."""
    features = {}
    positions = {}
    if model_name == "resnet50":
        h = model.maxpool(model.relu(model.bn1(model.conv1(x))))
        for name in LAYERS[model_name]:
            h = getattr(model, name)(h)
            row, col = h.shape[-2] // 2, h.shape[-1] // 2
            features[name] = h[0, :, row, col]
            positions[name] = {"grid_row": row, "grid_col": col,
                               "grid_height": h.shape[-2], "grid_width": h.shape[-1],
                               "token_index": -1}
    elif model_name == "vit_b_16":
        # Match torchvision's patch embedding -> CLS -> position -> encoder path.
        # Stop before encoder.ln: channel averaging after LayerNorm is degenerate.
        h = model._process_input(x)
        grid = model.image_size // model.patch_size
        row, col = grid // 2, grid // 2
        token_index = 1 + row * grid + col  # +1 skips the CLS token.
        h = torch.cat([model.class_token.expand(x.shape[0], -1, -1), h], dim=1)
        h = model.encoder.dropout(h + model.encoder.pos_embedding)
        for depth, block in enumerate(model.encoder.layers, start=1):
            h = block(h)
            name = f"block{depth}"
            if name in LAYERS[model_name]:
                features[name] = h[0, token_index, :]
                positions[name] = {"grid_row": row, "grid_col": col,
                                   "grid_height": grid, "grid_width": grid,
                                   "token_index": token_index}
    else:
        raise ValueError(f"Unsupported model: {model_name}")
    return features, positions


@torch.enable_grad()
def compute_erf(model, model_name, normalized_image, std):
    """Differentiate center feature energy with respect to RGB crop in [0,1]."""
    if torch.is_inference_mode_enabled():
        raise RuntimeError("Run ERF outside torch.inference_mode().")
    if normalized_image.shape != (1, 3, 224, 224):
        raise ValueError("Expected one normalized RGB image: [1, 3, 224, 224].")
    if len(std) != 3 or any(value <= 0 for value in std):
        raise ValueError("Expected three positive RGB normalization std values.")
    model.eval()  # Disable dropout; use saved BatchNorm statistics.
    x = normalized_image.detach().clone().requires_grad_(True)
    features, positions = center_features(model, model_name, x)
    scale = x.new_tensor(std).view(1, 3, 1, 1)
    maps = {}
    for stage, (name, vector) in enumerate(features.items(), start=1):
        # One explicit scalar for both architectures. This is a sensitivity probe,
        # not an exact all-channel Jacobian norm or a class attribution.
        z = vector.square().mean()
        gradient = torch.autograd.grad(
            z, x, retain_graph=stage < len(features), create_graph=False,
        )[0]
        # x_norm=(RGB-mean)/std, so dz/dRGB = (dz/dx_norm)/std.
        raw = (gradient / scale).abs().sum(dim=1)[0].detach().cpu().numpy()
        raw = raw.astype(np.float32)
        if not np.isfinite(raw).all() or float(raw.sum()) <= 0:
            raise ValueError(f"Zero or non-finite gradient for {model_name}/{name}.")
        maps[name] = {"raw": raw, "stage": stage, "target_value": z.item(),
                      **positions[name]}
    return maps

## Bước 5 — Đo ERF bằng số, không chỉ nhìn màu

- **A50 / A90:** sắp pixel theo gradient giảm dần, lấy số pixel nhỏ nhất chứa
  ít nhất 50% / 90% tổng gradient rồi chia cho 224×224. Pixel có thể rời nhau;
  đây không phải bán kính, bounding box hay diện tích vùng liên thông.
- **Dispersion:** khoảng cách RMS tới trọng tâm phân bố gradient, dùng tọa độ
  x,y chuẩn hóa [0,1]. Cao hơn nghĩa là ảnh hưởng phân tán xa trọng tâm hơn.
- **Center / outer energy:** tỷ lệ gradient trong / ngoài hình vuông giữa ảnh,
  có cạnh bằng nửa chiều ảnh. Tâm ở đây là tâm ảnh, không phải vị trí feature.

Ví dụ minh họa: gradient đều trên toàn ảnh → A90 gần 90%; gradient chỉ tại
một pixel → A90 bằng 1/(224×224). Không suy ra model nào tốt hơn từ A90.

In [ ]:
def erf_metrics(raw):
    """Area = smallest NUMBER of pixels reaching a fraction of absolute gradient."""
    raw = np.asarray(raw, dtype=np.float64)
    if raw.ndim != 2 or not np.isfinite(raw).all() or (raw < 0).any() or raw.sum() <= 0:
        raise ValueError("ERF must be a finite, nonnegative 2D map with positive sum.")
    p = raw / raw.sum()
    ranked = np.sort(p.ravel())[::-1]
    cumulative = np.cumsum(ranked)

    def area(q):
        count = min(np.searchsorted(cumulative, q, side="left") + 1, p.size)
        return float(count / p.size)

    yy, xx = np.indices(p.shape, dtype=np.float64)
    yy /= max(p.shape[0] - 1, 1)
    xx /= max(p.shape[1] - 1, 1)
    cy, cx = (p * yy).sum(), (p * xx).sum()
    dispersion = np.sqrt((p * ((yy - cy) ** 2 + (xx - cx) ** 2)).sum())
    # Central square occupies the middle half along each axis (~25% area).
    center = (np.abs(yy - 0.5) <= 0.25) & (np.abs(xx - 0.5) <= 0.25)
    center_energy = p[center].sum()
    return {
        "area50_fraction": area(0.5), "area90_fraction": area(0.9),
        "dispersion_normalized": float(dispersion),
        "center_energy_fraction": float(center_energy),
        "outer_energy_fraction": float(1 - center_energy),
        "centroid_y": float(cy), "centroid_x": float(cx),
        "raw_gradient_sum": float(raw.sum()),
    }

In [ ]:
# Chỉ kiểm tra định nghĩa metric bằng mảng giả, không phải kết quả ImageNet.
uniform = np.ones((10, 10))
point = np.zeros((10, 10))
point[5, 5] = 1
print('Đều trên 100 pixel → A90:', erf_metrics(uniform)['area90_fraction'])
print('Tập trung 1/100 pixel → A90:', erf_metrics(point)['area90_fraction'])

In [ ]:
def plot_image_maps(image_id, previews, destination=None):
    """Show each model's own crop and four ERF maps; color is log10 probability."""
    fig, axes = plt.subplots(2, 5, figsize=(16, 6), constrained_layout=True)
    for row, model_name in enumerate(MODEL_SPECS):
        crop, maps = previews[model_name]
        axes[row, 0].imshow(crop)
        axes[row, 0].set_title(f"{model_name}: RGB crop")
        for col, (layer, result) in enumerate(maps.items(), start=1):
            raw = result["raw"]
            probability = raw / raw.sum()
            heat = axes[row, col].imshow(
                np.log10(probability + 1e-12), cmap="magma", vmin=-8, vmax=-2,
                interpolation="nearest",
            )
            area90 = erf_metrics(raw)["area90_fraction"]
            axes[row, col].set_title(f"{layer}\nA90 = {area90:.1%}")
        for axis in axes[row]:
            axis.axis("off")
    fig.colorbar(heat, ax=axes[:, 1:].ravel().tolist(), shrink=0.7,
                 label="log10(normalized absolute gradient + 1e-12)")
    fig.suptitle(image_id)
    if destination is not None:
        fig.savefig(destination, dpi=160)
    return fig


def plot_summary(metrics, destination=None):
    """Error bars are across-image standard deviations, not confidence intervals."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    for model_name, group in metrics.groupby("model", sort=False):
        grouped = group.groupby("stage")
        for axis, metric, title in zip(
            axes, ["area90_fraction", "dispersion_normalized"],
            ["Smallest pixel area containing 90% gradient mass", "Spatial dispersion"],
        ):
            mean = grouped[metric].mean()
            sd = grouped[metric].std().fillna(0)
            axis.errorbar(mean.index, mean, yerr=sd, marker="o", capsize=3,
                          label=model_name)
            axis.set(title=title, xlabel="Relative stage (not equivalent layers)",
                     xticks=[1, 2, 3, 4])
            axis.grid(alpha=0.2)
            axis.legend()
    axes[0].set_ylabel("Fraction of input pixels (mean ± SD)")
    axes[1].set_ylabel("Normalized-coordinate RMS distance (mean ± SD)")
    if destination is not None:
        fig.savefig(destination, dpi=160)
    return fig

## Bước 6 — Chạy đúng một ảnh trước

Ô dưới tải lần lượt hai bộ trọng số từ cache hoặc máy chủ PyTorch. Chỉ giữ một
model trên GPU mỗi lần; batch size bằng 1. Đóng băng trọng số không ngăn gradient
theo ảnh đầu vào. Lần tải trọng số đầu tiên cần internet.

Mỗi hàng hiển thị crop model thực sự nhận và 4 ERF maps. Cùng thang màu log10
cho mọi bản đồ để nhìn được gradient yếu. Màu bị giới hạn để hiển thị;
A90 tính từ gradient gốc, không từ màu hay bản đồ đã clipping.

Preprocessing chính giữ thiết lập nhóm: ResNet resize cạnh ngắn 232, ViT 256,
rồi center crop 224. **Cùng ảnh gốc nhưng hai crop khác nhau.**
Vì vậy khác biệt ERF phản ánh cả model và preprocessing. Sau thí nghiệm chính,
đặt `PREPROCESSING='shared_vit_crop'` để chạy thêm kiểm tra với cùng crop 256→224;
chế độ phụ này thay preprocessing chính thức của ResNet và phải báo cáo riêng.

In [ ]:
def run_experiment(records, data_dir, output_dir, device, seed=42,
                   preprocessing="official", num_previews=3):
    """One model at a time; save raw maps, per-image metrics, and provenance."""
    if preprocessing not in {"official", "shared_vit_crop"}:
        raise ValueError("preprocessing must be official or shared_vit_crop.")
    # Preflight dataset BEFORE creating output files or downloading model weights.
    loader = load_images(data_dir, records)
    first = next(loader)
    loader.close()
    del first
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=False)  # Never mix runs or overwrite results.
    (output / "maps").mkdir()
    (output / "figures").mkdir()
    pd.DataFrame(records).to_csv(output / "selected_images.csv", index=False)
    torch.manual_seed(seed)
    np.random.seed(seed)
    metadata = {
        "status": "running", "num_images": len(records), "seed": seed,
        "device": str(device), "torch": torch.__version__,
        "torchvision": torchvision.__version__, "numpy": np.__version__,
        "preprocessing": preprocessing,
        "target": "mean(square(center_feature_vector)); post-stage/post-block, pre-final-LN",
        "center_rule": "row=grid_height//2, col=grid_width//2; zero-based",
        "gradient_coordinates": "RGB [0,1] AFTER resize/crop, BEFORE normalization",
        "layers": LAYERS, "models": {},
        "selected_images_sha256": hashlib.sha256(
            (output / "selected_images.csv").read_bytes()).hexdigest(),
        "source_sha256": hashlib.sha256(
            Path(__file__).read_bytes()).hexdigest() if "__file__" in globals() else None,
    }
    metadata_path = output / "run.json"

    def save_metadata():
        metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    save_metadata()
    rows, previews = [], {}
    preview_ids = {row["image_id"] for row in records[:num_previews]}
    model = None
    try:
        for model_name, (factory, weights) in MODEL_SPECS.items():
            transform_weights = (ViT_B_16_Weights.IMAGENET1K_V1
                                 if preprocessing == "shared_vit_crop" else weights)
            transform = transform_weights.transforms()
            metadata["models"][model_name] = {
                "weights": str(weights), "transform": repr(transform),
            }
            save_metadata()
            model = factory(weights=weights).to(device).eval()
            model.requires_grad_(False)  # Input gradient still works; no training.
            for record, image in tqdm(load_images(data_dir, records),
                                      total=len(records), desc=model_name):
                x = transform(image).unsqueeze(0).to(device)
                maps = compute_erf(model, model_name, x, transform.std)
                image_id = record["image_id"]
                # Every raw map is retained, including maps not selected for preview.
                np.savez_compressed(
                    output / "maps" / f"{model_name}_{record['dataset_index']:05d}.npz",
                    **{name: result["raw"] for name, result in maps.items()},
                )
                for layer, result in maps.items():
                    rows.append({**record, "model": model_name, "layer": layer,
                                 **{k: v for k, v in result.items() if k != "raw"},
                                 **erf_metrics(result["raw"])})
                if image_id in preview_ids:
                    rgb = (x[0].detach().cpu() * torch.tensor(transform.std)[:, None, None]
                           + torch.tensor(transform.mean)[:, None, None])
                    previews.setdefault(image_id, {})[model_name] = (
                        rgb.permute(1, 2, 0).clamp(0, 1).numpy(), maps,
                    )
                # Checkpoint CSV after each image; an interrupted run stays marked running.
                pd.DataFrame(rows).to_csv(output / "metrics.csv", index=False)
                del x, maps
            del model
            model = None
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        metrics = pd.DataFrame(rows)
        expected_rows = len(records) * sum(map(len, LAYERS.values()))
        if len(metrics) != expected_rows:
            raise ValueError(f"Expected {expected_rows} metrics rows, got {len(metrics)}.")
        summary = metrics.groupby(["model", "stage", "layer"])[[
            "area50_fraction", "area90_fraction", "dispersion_normalized",
            "center_energy_fraction", "outer_energy_fraction",
        ]].agg(["mean", "std", "count"])
        summary.to_csv(output / "summary.csv")
        for index, (image_id, preview) in enumerate(previews.items()):
            fig = plot_image_maps(image_id, preview, output / "figures" / f"image_{index:02d}.png")
            plt.close(fig)
        fig = plot_summary(metrics, output / "figures" / "comparison.png")
        plt.close(fig)
        metadata["status"] = "complete"
        save_metadata()
        return metrics, summary, previews
    except Exception as error:
        metadata.update(status="failed", error=str(error))
        save_metadata()
        raise
    finally:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
from datetime import datetime, timezone

# Đường dẫn có thời gian để không ghi đè kết quả cũ.
def new_run_dir(name):
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    return OUTPUT_BASE / f'{name}_{PREPROCESSING}_{stamp}'

smoke_dir = new_run_dir('smoke_1')
smoke_metrics, smoke_summary, smoke_previews = run_experiment(
    records[:1], DATA_DIR, smoke_dir, DEVICE,
    seed=SEED, preprocessing=PREPROCESSING, num_previews=1,
)
display(smoke_metrics[['model', 'layer', 'grid_row', 'grid_col', 'token_index',
                       'area50_fraction', 'area90_fraction', 'dispersion_normalized']])
fig = plot_image_maps(records[0]['image_id'], smoke_previews[records[0]['image_id']])
plt.show()
print('Đã lưu:', smoke_dir)

**Trước khi chạy nhiều ảnh:** xác nhận có 8 dòng metrics (2 model × 4 tầng),
gradient hữu hạn và tổng dương, A50 ≤ A90 ≤ 1. Nếu lỗi thiếu shard, quay lại
bước 1. Nếu GPU hết bộ nhớ, khởi động lại runtime để giải phóng model còn sót
từ các lần chạy trước; notebook đã dùng từng ảnh và từng model.

Không kết luận xu hướng từ một ảnh. Ta chuyển sang nhiều ảnh để tính phân bố.

In [ ]:
assert len(smoke_metrics) == 8
assert np.isfinite(smoke_metrics['raw_gradient_sum']).all()
assert (smoke_metrics['raw_gradient_sum'] > 0).all()
assert (smoke_metrics['area50_fraction'] <= smoke_metrics['area90_fraction']).all()
assert smoke_metrics['area90_fraction'].between(0, 1).all()
print('Kiểm tra một ảnh đạt. Có thể chuyển sang tập nhỏ.')

## Bước 7 — Chạy tập nhỏ, rồi mở rộng

Ô này chạy `NUM_IMAGES` đã chọn ở bước 2. Sau khi chạy ổn với 10 ảnh,
đổi thành 100–200, chạy lại ô chọn `records` rồi chạy lại ô dưới.
Không cần chạy lại baseline hay train model. Ta giữ tất cả ảnh được chọn,
không lọc theo dự đoán đúng/sai.

CSV có 8×N dòng; NPZ lưu các bản đồ gradient thô cho từng model/ảnh.
`summary.csv` tổng hợp metric từng ảnh: **không** tính A90 của heatmap trung bình.
Thanh sai số trên biểu đồ là độ lệch chuẩn giữa ảnh, không phải khoảng tin cậy.
Khi một ảnh duy nhất, SD không xác định; biểu đồ hiển thị thanh sai số bằng 0.

In [ ]:
run_dir = new_run_dir(f'main_{len(records)}')
metrics, summary, previews = run_experiment(
    records, DATA_DIR, run_dir, DEVICE,
    seed=SEED, preprocessing=PREPROCESSING, num_previews=3,
)
display(summary)
fig = plot_summary(metrics)
plt.show()
print('Hoàn tất:', run_dir)
print('Số dòng metrics:', len(metrics), '| Mong đợi:', 8 * len(records))

## Bước 8 — Đọc và viết kết quả

Bảng dưới so sánh trên **cùng từng ảnh**, với chênh lệch A90 = ViT − ResNet.
Giá trị dương nghĩa là ViT cần nhiều pixel hơn để chứa 90% gradient theo phép đo này.
Chỉ ghép theo giai đoạn tương đối; không khẳng định layer1 tương đương block3.

In [ ]:
paired = metrics.pivot(index=['image_id', 'stage'], columns='model', values='area90_fraction')
paired['vit_minus_resnet_pp'] = 100 * (paired['vit_b_16'] - paired['resnet50'])
paired_summary = paired.groupby('stage')['vit_minus_resnet_pp'].agg(['mean', 'std', 'count'])
paired_summary.to_csv(run_dir / 'paired_area90_difference.csv')
display(paired_summary)

**Mẫu viết báo cáo — điền sau khi có số liệu thật:**

> Chúng tôi phân tích [N] ảnh lấy từ manifest 10.000 ảnh, seed [seed], dùng
> ResNet-50 IMAGENET1K_V2 và ViT-B/16 IMAGENET1K_V1. ERF được ước lượng bằng
> tổng trị tuyệt đối gradient theo RGB của năng lượng vector đặc trưng ở gần tâm.
> Tại giai đoạn [s], A90 trung bình là [...] ở ResNet và [...] ở ViT;
> chênh lệch ghép cặp trung bình là [...] điểm phần trăm. Bản đồ [...] cho thấy [...].

**Discussion cần trả lời:**

1. A90 và dispersion thay đổi thế nào từ tầng đầu tới cuối? Các ảnh có đồng nhất không?
2. Khác biệt còn giữ khi dùng cùng crop không? Có trường hợp ngược xu hướng không?
3. Liên hệ với attention/Grad-CAM/occlusion của nhóm, không suy ra quan hệ nhân quả.
4. Nêu giới hạn: tập con nhỏ; chọn một vị trí và một scalar; gradient là độ nhạy
   cục bộ; trọng số và cách huấn luyện khác nhau; crop và lưới đặc trưng khác nhau.

Baseline nhóm: Top-1 80,844% và 81,066%, chênh 0,222 điểm phần trăm.
Báo cáo độ chênh thực tế; nhóm chưa đặt ngưỡng matched accuracy.
Không viết “ViT luôn có ERF lớn hơn” hoặc “ERF rộng chứng minh model tốt hơn”.

**Sản phẩm bàn giao trong thư mục run:**

- `selected_images.csv`: đúng ảnh/nhãn/index đã dùng.
- `metrics.csv`, `summary.csv`, `paired_area90_difference.csv`: số liệu.
- `maps/*.npz`: gradient thô từng tầng, có thể chuẩn hóa lại từ đây.
- `figures/*.png`: so sánh theo ảnh và biểu đồ theo giai đoạn.
- `run.json`: trạng thái, phiên bản, trọng số, preprocessing, vị trí và target.

Khi gián đoạn, `run.json` ở trạng thái running hoặc failed và CSV có thể chưa đủ.
Chạy lại tạo thư mục mới; chưa có chức năng resume. Generated outputs đang được
`.gitignore` loại ra. Đừng ép thêm ảnh ImageNet, weights hoặc NPZ vào Git.
Trước khi chia sẻ notebook, xóa output các ô hiển thị ảnh ImageNet.

## Tài liệu tham khảo

1. Luo et al. (2016), [Understanding the Effective Receptive Field in Deep Convolutional Neural Networks](https://papers.neurips.cc/paper_files/paper/2016/hash/c8067ad1937f728f51288b3eb986afaa-Abstract.html).
2. Raghu et al. (2021), [Do Vision Transformers Act Like Convolutional Neural Networks?](https://research.google/pubs/do-vision-transformers-act-like-convolutional-neural-networks/).
3. Torchvision: [ResNet-50 weights/transforms](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html),
   [ViT-B/16 weights/transforms](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vit_b_16.html),
   [ViT implementation](https://docs.pytorch.org/vision/stable/_modules/torchvision/models/vision_transformer.html).
4. [Colab FAQ: GPU availability, runtimes, and Drive](https://research.google.com/colaboratory/faq.html).